# System Benchmark v2.1 — Modern Data Lakehouse

Notebook này đo **4 metrics chính** sau khi toàn bộ pipeline đã chạy xong:

| # | Metric | Phương pháp |
|---|---|---|
| 1 | **End-to-end Ingestion Latency** | Đọc từ Bronze snapshot history — khoảng thời gian giữa 2 batch commit liên tiếp |
| 2 | **SCD2 MERGE Throughput** | Đọc từ Silver snapshot history — added/deleted records per second |
| 3 | **Compaction Impact** | Số file + query time trước/sau REWRITE DATA FILES |
| 4 | **Gold Query Latency** | Thời gian analytics query trên Gold layer |

> **v2 Fix:** Metric 1 & 2 không dùng probe/polling hay MERGE trực tiếp nữa.
> Thay vào đó đọc từ Iceberg snapshot metadata — hoạt động ngay cả khi benchmark
> đang reuse SparkSession của Bronze/Silver (cùng JVM).
>
> **v2.1 Fix (sau review):**
> - Metric 2 lọc bỏ snapshot `operation = 'replace'` (do compaction) khỏi snapshot
>   history, để benchmark vẫn đúng nếu chạy lại nhiều lần mà không reset pipeline.
> - Metric 3 thêm warm-up **đối xứng** cho cả lần đo "trước" và "sau" compaction,
>   tránh trường hợp % speedup bị lẫn hiệu ứng cache warm-up.

Kết quả xuất ra bảng tổng hợp cuối notebook — copy thẳng vào chương **"Đánh giá hệ thống"** của luận văn.

> Notebook đã module hóa — toàn bộ logic core nằm trong package `src/benchmark.py`.
> Notebook chỉ setup SparkSession, gọi function, và in kết quả.


In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.spark_session import get_spark_session
from src import benchmark

spark = get_spark_session("Lakehouse_Benchmark")
results = {}

print("✅ SparkSession ready — bắt đầu benchmark v2.1")


✅ SparkSession ready — bắt đầu benchmark v2.1


26/06/28 14:14:31 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


---
## Metric 1 — End-to-end Ingestion Latency (Kafka → Bronze)

**Phương pháp v2:** Đọc từ Bronze snapshot history.
Mỗi lần Bronze stream trigger (15s) thì Iceberg commit 1 snapshot mới.
Khoảng thời gian giữa 2 snapshot liên tiếp = batch interval thực tế của pipeline.

Không cần stream đang chạy, không block session — đọc hoàn toàn từ metadata.


In [2]:
print("=" * 55)
print("METRIC 1: END-TO-END INGESTION LATENCY")
print("=" * 55)

results.update(benchmark.measure_ingestion_latency(spark))


METRIC 1: END-TO-END INGESTION LATENCY
  Tổng số snapshots Bronze : 159

  Batch interval (avg)     : 15.4s
  Batch interval (median)  : 15.0s  ← dùng giá trị này cho luận văn
  Batch interval (min)     : 2.5s
  Batch interval (max)     : 73.7s

  Trigger config           : 15s
  → End-to-end latency ≈ median batch interval = 15.0s
    (record vào Kafka → commit vào Iceberg Bronze)

  5 snapshots gần nhất:
    13:57:30 | +  5,700 records | total:  2,759,314
    13:57:45 | +  5,912 records | total:  2,765,226
    13:58:00 | +  6,512 records | total:  2,771,738
    13:58:15 | +  5,850 records | total:  2,777,588
    13:58:30 | +  5,467 records | total:  2,783,055


---
## Metric 2 — SCD2 MERGE Throughput (Silver Layer)

**Phương pháp v2:** Đọc từ Silver snapshot history.
Mỗi micro-batch của Silver tạo ra 2 snapshot (MERGE step 1 + step 2).
Throughput = tổng records được xử lý / tổng thời gian pipeline chạy.

> v2.1: loại snapshot `operation = 'replace'` (compaction) để không bị méo số khi chạy lại.


In [3]:
print("=" * 55)
print("METRIC 2: SCD2 MERGE THROUGHPUT")
print("=" * 55)

results.update(benchmark.measure_silver_throughput(spark))


METRIC 2: SCD2 MERGE THROUGHPUT


  Total records in Silver  : 2,783,055
  Unique users (SCD keys)  : 1,987,897
  Active records (current) : 1,987,897
  Expired (SCD2 history)   : 795,158 (28.6% of total)

  Silver snapshot history (đã loại snapshot compaction):
    Total snapshots          : 220  (2 per micro-batch = MERGE step1 + step2)
    Total added ops          : 174,864,776
    Total deleted ops        : 172,081,721
    Pipeline duration        : 2237s (37.3 phút)
    Throughput (records/s)   : 1,244.2  ← dùng giá trị này cho luận văn
    MERGE ops/s (add+del)    : 155,101.0

  MERGE latency per micro-batch (từ Silver snapshot intervals):
    Số micro-batch đo được   : 110
    MERGE latency (avg)      : 1874 ms
    MERGE latency (median)   : 1643 ms  ← dùng giá trị này cho luận văn


---
## Metric 3 — Compaction Impact (Iceberg REWRITE DATA FILES)

Streaming sinh ra nhiều small files — đây là vấn đề phổ biến trong mọi hệ thống streaming.
Compaction gộp các small files thành ít file lớn hơn, cải thiện read performance đáng kể.


In [4]:
print("=" * 55)
print("METRIC 3A: FILE STATS TRƯỚC COMPACTION")
print("=" * 55)

before_raw, before_metrics = benchmark.measure_files_before_compaction(spark)
results.update(before_metrics)


METRIC 3A: FILE STATS TRƯỚC COMPACTION
  [Bronze] Files: 159 | Total: 83.1 MB | Avg: 535.1 KB | Min: 13.0 KB
  [Silver] Files: 364 | Total: 85.4 MB | Avg: 240.3 KB | Min: 4.8 KB
  [Gold] Files: 12 | Total: 11.4 MB | Avg: 968.8 KB | Min: 610.9 KB

  [Warm-up] Đang flush cache...
  [Warm-up] Done — bắt đầu đo...

  Query time trước compaction:


  Bronze scan     : 799 ms (median of 3 runs)


  Silver aggregate: 1605 ms (median of 3 runs)
  Gold aggregate  : 472 ms (median of 3 runs)


In [5]:
print("=" * 55)
print("METRIC 3B: COMPACTION — REWRITE DATA FILES")
print("=" * 55)

results.update(benchmark.run_compaction(spark))


METRIC 3B: COMPACTION — REWRITE DATA FILES
  [Bronze] Đang compact...


+--------------------------+----------------------+---------------------+-----------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|
+--------------------------+----------------------+---------------------+-----------------------+
|                       159|                     1|             87117250|                      0|
+--------------------------+----------------------+---------------------+-----------------------+

  [Bronze] Compaction xong: 11.6s

  [Silver] Đang compact...


+--------------------------+----------------------+---------------------+-----------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|
+--------------------------+----------------------+---------------------+-----------------------+
|                       364|                     1|             89583624|                      0|
+--------------------------+----------------------+---------------------+-----------------------+

  [Silver] Compaction xong: 9.5s

  [Gold] Đang compact...


+--------------------------+----------------------+---------------------+-----------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|
+--------------------------+----------------------+---------------------+-----------------------+
|                        12|                     1|             11905192|                      0|
+--------------------------+----------------------+---------------------+-----------------------+

  [Gold] Compaction xong: 3.2s



In [6]:
print("=" * 55)
print("METRIC 3C: FILE STATS SAU COMPACTION")
print("=" * 55)

results.update(benchmark.measure_files_after_compaction(spark, before_raw))


METRIC 3C: FILE STATS SAU COMPACTION

  Query time sau compaction:
  [Warm-up] Đang flush cache...
  [Warm-up] Done — bắt đầu đo...
  Bronze scan     : 176 ms (median of 3 runs)
  [Bronze] Files: 159 → 1 (+99%) | Query: 799ms → 176ms
  Silver aggregate: 275 ms (median of 3 runs)
  [Silver] Files: 364 → 1 (+100%) | Query: 1605ms → 275ms
  Gold aggregate  : 194 ms (median of 3 runs)
  [Gold] Files: 12 → 1 (+92%) | Query: 472ms → 194ms


---
## Metric 4 — Gold Query Latency (Analytics Queries)


In [7]:
print("=" * 55)
print("METRIC 4: GOLD LAYER QUERY LATENCY")
print("=" * 55)

results.update(benchmark.measure_gold_query_latency(spark))


METRIC 4: GOLD LAYER QUERY LATENCY
  Q1_count_scan: 97 ms (median of 3 runs)
  Q2_group_by_state: 181 ms (median of 3 runs)
  Q3_filter_multi_attr: 175 ms (median of 3 runs)
  Q4_category_explode: 290 ms (median of 3 runs)


  Q5_scd2_history_join: 3128 ms (median of 3 runs)

  Chú thích:
  Q1: Full scan + count (đo I/O baseline)
  Q2: GROUP BY state (đo aggregation)
  Q3: Multi-column filter (đo predicate pushdown)
  Q4: LATERAL VIEW explode array (đo array processing)
  Q5: SCD2 history query — full scan Silver + GROUP BY + HAVING.
      Latency cao hơn Q1-Q4 là trade-off bình thường của SCD2 pattern:
      mỗi user_id có nhiều version row → không thể predicate pushdown.
      Đây là chi phí của việc lưu lịch sử thay đổi (history preservation).


---
## Tổng hợp kết quả — Bảng cho Luận văn


In [8]:
save_path = str(project_root / "benchmark_results.json")
benchmark.print_summary(results, save_path)



╔═════════════════════════════════════════════════════════════════╗
║  KẾT QUẢ BENCHMARK — MODERN DATA LAKEHOUSE                      ║
╠═════════════════════════════════════════════════════════════════╣
║  METRIC 1: END-TO-END INGESTION LATENCY (Kafka → Bronze)        ║
║    Batch interval (median)  : 15.0 giây                         ║
║    Batch interval (avg)     : 15.4 giây                         ║
║    Batch interval (min/max) : 2.5s / 73.7s                      ║
║    Trigger config           : 15s                               ║
╠═════════════════════════════════════════════════════════════════╣
║  METRIC 2: SILVER SCD2 THROUGHPUT                               ║
║    Total records Silver     : 2,783,055                         ║
║    Active (is_current=true) : 1,987,897                         ║
║    Expired (SCD2 history)   : 795,158                           ║
║    Pipeline duration        : 2237s                             ║
║    Avg throughput           : 1244.2 records/